# 1 — Setup and Configuration
**Image Enhancement as Preprocessing for Industrial Anomaly Detection Under Challenging Illumination.** Exploratory screening of all eight MVTec AD 2 categories (or an explicit subset); four conditions; PaDiM with fixed pretrained ResNet18. This notebook relies on Carlos's accepted dataset qualification and does not repeat validation or change inputs.
Select the workload below, then use **Runtime → Run all** in a fresh Colab runtime. CPU works; an available GPU accelerates the frozen backbone. ImageNet weights may download on first use. Only generated evidence is written to Drive. Each run gets a new UTC folder and begins with **incomplete** status.
Small screening informs follow-up scope; it does not select final categories or establish a winning method. Batch 2 preserves the Batch 1 scientific configuration and nearest-neighbor mask policy. UTC logs and structured stage timings record actual executed stages.


In [ ]:
# Configure workload before dataset reading, model loading, or feature extraction.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import csv
import json
import random
import time
import platform
import numpy as np
import scipy
from scipy.ndimage import label as connectedComponents, gaussian_filter
import sklearn
from sklearn.metrics import roc_auc_score, roc_curve, auc
import cv2
import PIL
from PIL import Image
import torch
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
from torch.nn import functional as F

workloadSize = 'small'  # 'small', 'medium', 'large'
randomSeed = 7367
allCategories = ['can', 'fabric', 'fruit_jelly', 'rice', 'sheet_metal', 'vial', 'wallplugs', 'walnuts']
selectedCategories = list(allCategories)  # Or an explicit subset, e.g. ['can', 'wallplugs'].
if (not selectedCategories or len(selectedCategories) != len(set(selectedCategories))
        or not set(selectedCategories).issubset(allCategories)):
    raise ValueError('selectedCategories must be a nonempty, unique subset of the eight MVTec AD 2 categories.')
workloadLimits = {'small': (25, 20), 'medium': (250, 100), 'large': (None, None)}
if workloadSize not in workloadLimits:
    raise ValueError("workloadSize must be 'small', 'medium', or 'large'.")
referenceCap, evaluationCap = workloadLimits[workloadSize]
datasetRoot = Path('/content/drive/MyDrive/ColabData/CS7367_Vision/mvtec_ad_2')
outputRoot = Path('/content/drive/MyDrive/ColabData/CS7367_Vision/output/experiments')
categories = selectedCategories
conditions = ['baseline', 'gamma', 'histogram_equalization', 'clahe']
imageSize = 256
featureGrid = 32
featureDimension = 32
batchSize = 5
covarianceRegularization = 0.01
enhancementParameters = {'gamma': 0.7, 'claheClipLimit': 2.0, 'claheTileGrid': [8, 8],
                         'channel': 'CIELAB L after RGB bilinear resize; baseline RGB unchanged'}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(randomSeed)
np.random.seed(randomSeed)
torch.manual_seed(randomSeed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(randomSeed)
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)
runStarted = time.perf_counter()
runTimestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
runRoot = outputRoot / workloadSize / runTimestamp
if runRoot.resolve().is_relative_to(datasetRoot.resolve()):
    raise ValueError('Outputs must be outside the original dataset.')
runRoot.mkdir(parents=True, exist_ok=False)  # Never overwrite an existing run.
metricFields = ['dataset', 'category', 'condition', 'workloadMode', 'referenceCount',
                'evaluationCount', 'normalEvaluationCount', 'anomalyEvaluationCount',
                'metricName', 'metricValue', 'metricStatus', 'runtimeSeconds']
metricRows = []
manifestRows = []
selectedData = {}
summary = dict(implementationVersion='research-batch2', purpose='exploratory category screening; no final category or winning-method claim',
               stageTimings=[], maskDiagnostics=[], maskResamplingPolicy='Pillow NEAREST to 256x256, unchanged from Batch 1; any lost anomaly suppresses localization for that category across all conditions',
               comparisonPolicy='Existing-category results retain Batch 1 methodology; compare identical manifests/settings. Added categories are exploratory.',
               status='incomplete', workloadMode=workloadSize, configuredCaps={'reference': referenceCap, 'evaluation': evaluationCap},
               seed=randomSeed, utcTimestamp=runTimestamp, selectedCategories=categories,
               splitPolicy='Fit train/good only; evaluate test_public/good and test_public/bad only; validation and private tests excluded.',
               enhancementParameters=enhancementParameters, conditions=conditions,
               model={'name': 'PaDiM', 'backbone': 'ResNet18', 'weights': 'IMAGENET1K_V1', 'layers': ['layer1', 'layer2', 'layer3'],
                      'inputSize': imageSize, 'featureGrid': featureGrid, 'featureDimension': featureDimension,
                      'covarianceRegularization': covarianceRegularization, 'batchSize': batchSize, 'smoothingSigma': 4.0},
               metricPolicy='Image AUROC on public labels; normalized AU-PRO0.05 at 256x256, 8-connected regions, all selected public masks required.',
               environment={'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__,
                            'numpy': np.__version__, 'scipy': scipy.__version__, 'sklearn': sklearn.__version__,
                            'opencv': cv2.__version__, 'pillow': PIL.__version__, 'device': str(device)},
               configuredDatasetRoot=str(datasetRoot), outputPath=str(runRoot), actualCounts={}, metricAvailability=[],
               limitations=['Small/medium AUROC describes the selected balanced sample, not full-set AUROC.',
                            'Metrics use 256x256 images and masks; resizing can erase tiny defects and is not native-resolution benchmark certification.',
                            'No private-test illumination-shift metrics are inferred; public evaluation cannot replace private-server evaluation.',
                            'Fixed 32-dimensional features and 32x32 spatial pooling are a bounded PaDiM configuration, not a claimed reproduction of published scores.'],
               errors=[], totalRuntimeSeconds=0)

stageStarts = {}

def logStage(event, stage, category='all', condition='all', processed=None, total=None):
    nowUtc = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.%fZ')
    nowCounter = time.perf_counter()
    key = (stage, category, condition)
    if event == 'START':
        timing = dict(stage=stage, category=category, condition=condition, startUtc=nowUtc,
                      endUtc=None, elapsedSeconds=None, status='incomplete')
        stageStarts[key] = (nowCounter, timing)
        summary['stageTimings'].append(timing)
    elapsed = nowCounter - stageStarts[key][0] if key in stageStarts else nowCounter - runStarted
    if event == 'COMPLETE':
        _, timing = stageStarts.pop(key)
        timing.update(endUtc=nowUtc, elapsedSeconds=elapsed, status='complete')
    progress = '' if processed is None else f' | {processed}/{total} images'
    context = f' | mode={workloadSize} | run={runTimestamp} | device={device}' if stage == 'notebook' else ''
    print(f'[{nowUtc}] {event} | {category} | {condition} | {stage}{progress} | elapsed {elapsed:.1f}s{context}', flush=True)

def saveEvidence():
    logStage('START', 'evidence writing')
    summary['totalRuntimeSeconds'] = time.perf_counter() - runStarted
    with (runRoot / 'metrics.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=metricFields)
        writer.writeheader()
        writer.writerows(metricRows)
    temporaryPath = runRoot / 'experiment_summary.json.tmp'
    temporaryPath.write_text(json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')
    temporaryPath.replace(runRoot / 'experiment_summary.json')
    logStage('COMPLETE', 'evidence writing')
    # Persist the observed end timing after the evidence has actually been written.
    temporaryPath.write_text(json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')
    temporaryPath.replace(runRoot / 'experiment_summary.json')

logStage('START', 'notebook')
saveEvidence()


# 2 — Dataset Selection and Sampling
Use **train/good only** as the normal reference; validation is held out and unused. Evaluate only **test_public/good** and **test_public/bad**, whose directory labels are public. Private partitions are excluded. The selector caps both roles; large uses all eligible files. Evaluation selection rotates through shuffled nonempty classes, prioritizing both classes over original prevalence. One manifest is reused for all four conditions, with disjoint reference/evaluation identities. Missing groups or insufficient references are explicit limitations, not fabricated data.


In [ ]:
# Select identities once, before enhancement. No file contents are altered.
def selectImages(groups, cap, seed):
    rng = random.Random(str(seed))
    pools = {group: sorted(paths) for group, paths in sorted(groups.items()) if paths}
    groupNames = sorted(pools)
    rng.shuffle(groupNames)
    for pool in pools.values():
        rng.shuffle(pool)
    selected = []
    total = sum(map(len, pools.values())) if cap is None else cap
    while len(selected) < total and any(pools.values()):
        for group in groupNames:
            if pools[group] and len(selected) < total:
                selected.append((group, pools[group].pop()))
    return selected

logStage('START', 'dataset discovery and sampling')
try:
    if not datasetRoot.is_dir():
        raise FileNotFoundError(datasetRoot)
    if not any((datasetRoot / c).is_dir() for c in categories):
        candidates = sorted({p.parent for c in categories for p in datasetRoot.rglob(c) if p.is_dir()})
        if len(candidates) != 1:
            raise ValueError('Cannot uniquely locate the MVTec category root.')
        datasetRoot = candidates[0]
    summary['accessibleDatasetRoot'] = str(datasetRoot)
    for category in categories:
        categoryRoot = datasetRoot / category
        groups = {}
        for group in ['train/good', 'test_public/good', 'test_public/bad']:
            folder = categoryRoot / group
            groups[group] = sorted(p for p in folder.glob('*') if p.is_file() and p.suffix.lower() in ['.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'])
            if not groups[group]:
                summary['limitations'].append(f'{category}: unavailable or empty {group}.')
        references = selectImages({'good': groups['train/good']}, referenceCap, f'{randomSeed}|{category}|reference')
        evaluations = selectImages({'good': groups['test_public/good'], 'bad': groups['test_public/bad']}, evaluationCap, f'{randomSeed}|{category}|evaluation')
        if {p.resolve() for _, p in references} & {p.resolve() for _, p in evaluations}:
            raise ValueError(f'{category}: reference/evaluation identity leakage.')
        selectedData[category] = {'reference': references, 'evaluation': evaluations}
        counts = dict(reference=len(references), evaluation=len(evaluations),
                      normalEvaluation=sum(g == 'good' for g, _ in evaluations), anomalyEvaluation=sum(g == 'bad' for g, _ in evaluations))
        summary['actualCounts'][category] = counts
        for role, records in selectedData[category].items():
            for group, imagePath in records:
                manifestRows.append(dict(category=category, role=role, label=int(group == 'bad'), group=group,
                                         relativeImagePath=imagePath.relative_to(datasetRoot).as_posix(), workloadMode=workloadSize))
        logStage('INFO', f'selected reference={len(references)}, evaluation={len(evaluations)} (good={counts["normalEvaluation"]}, bad={counts["anomalyEvaluation"]})', category)
    with (runRoot / 'sample_manifest.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=['category', 'role', 'label', 'group', 'relativeImagePath', 'workloadMode'])
        writer.writeheader()
        writer.writerows(manifestRows)
    logStage('COMPLETE', 'dataset discovery and sampling')
    saveEvidence()
except BaseException as error:
    summary['errors'].append(f'Selection: {type(error).__name__}: {error}')
    saveEvidence()
    raise



# 3 — Image Enhancement
Resize RGB images to **256×256 without cropping**, then apply baseline, gamma (`Lout = Lin**0.7` on normalized luminance), global histogram equalization, or CLAHE (clip limit 2; 8×8 tiles) on the CIELAB **L** channel. Convert back to RGB and use ImageNet normalization. These fixed parameters apply to **both reference and evaluation** images. No parameters are tuned on test labels. Masks receive no enhancement; nearest-neighbor resizing is solely the evaluation coordinate transform.


In [ ]:
# These pure transformations produce new in-memory arrays; original files remain untouched.
def enhanceImage(rgb, condition):
    if condition == 'baseline':
        return rgb.copy()
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    lightness = lab[:, :, 0]
    if condition == 'gamma':
        lightness = np.rint(255 * (lightness.astype(np.float32) / 255) ** enhancementParameters['gamma']).astype(np.uint8)
    elif condition == 'histogram_equalization':
        lightness = cv2.equalizeHist(lightness)
    elif condition == 'clahe':
        lightness = cv2.createCLAHE(clipLimit=enhancementParameters['claheClipLimit'],
                                   tileGridSize=tuple(enhancementParameters['claheTileGrid'])).apply(lightness)
    else:
        raise ValueError(f'Unknown enhancement: {condition}')
    lab[:, :, 0] = lightness
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

logStage('START', 'enhancement setup')
normalizationMean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
normalizationStd = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
logStage('COMPLETE', 'enhancement setup')
# Actual enhancement/features are included in reference-fitting and inference timings.


# 4 — PaDiM Feature Extraction
Frozen ImageNet ResNet18 layers 1–3 produce 448 concatenated channels. Bilinear alignment to **32×32** and one seed-fixed **32-channel subset** bound memory and are identical for all categories, conditions, and workload modes. Fit one regularized multivariate Gaussian per spatial position, using only normal training features. Batch-Welford covariance accumulation avoids storing the full reference feature set; float64 covariance plus **0.01 I** stabilizes the small-sample fit. No supervised classifier is trained.
Sources: [PaDiM paper](https://arxiv.org/abs/2011.08785), [torchvision ResNet18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html). This bounded parameterization is not a claim to reproduce the paper's benchmark scores.


In [ ]:
# Load one fixed backbone; no random-weight fallback if pretrained loading fails.
logStage('START', 'pretrained backbone loading')
try:
    backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).eval().requires_grad_(False).to(device)
    featureIndices = torch.randperm(448, generator=torch.Generator().manual_seed(randomSeed))[:featureDimension].to(device)
    summary['model']['featureIndices'] = featureIndices.cpu().tolist()
    logStage('COMPLETE', 'pretrained backbone loading')
    saveEvidence()
except BaseException as error:
    summary['errors'].append(f'Backbone: {type(error).__name__}: {error}')
    saveEvidence()
    raise


def extractFeatures(paths, condition):
    inputs = []
    for imagePath in paths:
        with Image.open(imagePath) as image:
            rgb = np.asarray(image.convert('RGB').resize((imageSize, imageSize), Image.Resampling.BILINEAR))
        tensor = torch.from_numpy(enhanceImage(rgb, condition)).permute(2, 0, 1).float() / 255
        inputs.append((tensor - normalizationMean) / normalizationStd)
    with torch.inference_mode():
        x = torch.stack(inputs).to(device)
        x = backbone.maxpool(backbone.relu(backbone.bn1(backbone.conv1(x))))
        layer1 = backbone.layer1(x)
        layer2 = backbone.layer2(layer1)
        layer3 = backbone.layer3(layer2)
        aligned = [F.interpolate(layer, size=(featureGrid, featureGrid), mode='bilinear', align_corners=False)
                   for layer in [layer1, layer2, layer3]]
        embeddings = torch.cat(aligned, dim=1)[:, featureIndices]
    return embeddings.permute(0, 2, 3, 1).reshape(len(paths), -1, featureDimension).cpu().numpy().astype(np.float64)

def fitPadim(references, category, condition):
    logStage('START', 'reference fitting (enhancement/features/covariance)', category, condition, 0, len(references))
    count = 0
    mean = np.zeros((featureGrid ** 2, featureDimension), dtype=np.float64)
    secondMoment = np.zeros((featureGrid ** 2, featureDimension, featureDimension), dtype=np.float64)
    for start in range(0, len(references), batchSize):
        features = extractFeatures([p for _, p in references[start:start+batchSize]], condition)
        newCount = len(features)
        batchMean = features.mean(axis=0)
        centered = features - batchMean
        delta = batchMean - mean
        # Merge batch and running centered sums; no evaluation images enter this fit.
        secondMoment += np.einsum('npd,npe->pde', centered, centered)
        secondMoment += np.einsum('pd,pe->pde', delta, delta) * count * newCount / (count + newCount)
        mean += delta * newCount / (count + newCount)
        count += newCount
        if count % 5 == 0:
            logStage('PROGRESS', 'reference fitting (enhancement/features/covariance)', category, condition, count, len(references))
    covariance = secondMoment / (count - 1) + covarianceRegularization * np.eye(featureDimension)
    precision = np.linalg.inv(covariance)
    logStage('COMPLETE', 'reference fitting (enhancement/features/covariance)', category, condition, count, len(references))
    return mean, precision


# 5 — Anomaly Scoring and Evaluation
Score patches by Mahalanobis distance, upsample to 256×256, smooth with Gaussian σ=4 pixels, and use the map maximum as the image anomaly score. No per-image min–max normalization is applied. Image AUROC needs both public classes. AU-PRO0.05 additionally needs every selected anomaly mask to be readable, aligned, binary, and nonempty after nearest-neighbor resizing; absent normal masks represent known defect-free images.
**AU-PRO0.05:** identify 8-connected components, weight each foreground pixel by the inverse area of its region, sweep all distinct score thresholds with ties grouped, linearly interpolate at FPR=0.05, integrate only [0,0.05], and divide by 0.05. This is equal-region overlap, not pixel AUROC. All background pixels, including normal images, enter FPR. Missing or degenerate ground truth produces a blank metric with an explicit reason; no metric is invented. Evaluation is at 256×256, not native benchmark resolution; tiny defects may vanish. Full score/mask arrays are retained for only one category/condition at a time; reference features remain streamed.
[MVTec metric definition](https://link.springer.com/article/10.1007/s11263-026-02743-0), [OpenCV enhancement operations](https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html).

**Tiny-mask diagnosis and unchanged policy:** read-only inspection of `can/test_public/bad/002_shift_1.png` found matching 2232×1024 image/mask dimensions, binary mask values {0,255}, and 25 native positive pixels. Pillow nearest-neighbor resize to 256×256 retained zero positive pixels. The native annotation is not empty; the fixed grid misses this small defect. Retain nearest-neighbor and category-level localization unavailability rather than dropping that image, dilating ground truth, or manufacturing pixels. The notebook saves native/resized areas, values and bounding boxes for any selected anomaly that vanishes. Image AUROC remains available when labels support it. A future higher/native-resolution protocol requires a separately approved, fresh comparison across all conditions; it is not applied automatically here.


In [ ]:
# Region-weighted ROC implements exact threshold sweeping with equal weight per defect region.
def auPro005(masks, scores):
    if masks.shape != scores.shape or masks.ndim != 3 or not np.isfinite(scores).all():
        return None, 'unavailable: invalid mask/score shapes or nonfinite scores'
    if not np.isin(masks, [0, 1]).all():
        return None, 'unavailable: masks must be binary'
    foreground = masks.astype(bool)
    if not foreground.any() or foreground.all():
        return None, 'unavailable: foreground regions and background pixels are required'
    regionWeights = np.ones(masks.shape, dtype=np.float64)
    for index, mask in enumerate(foreground):
        regions, regionCount = connectedComponents(mask, structure=np.ones((3, 3)))
        if regionCount:
            sizes = np.bincount(regions.ravel())
            regionWeights[index][mask] = 1.0 / sizes[regions[mask]]
    fpr, pro, _ = roc_curve(foreground.ravel(), scores.ravel(), sample_weight=regionWeights.ravel(), drop_intermediate=False)
    limit = 0.05
    inside = fpr < limit
    clippedFpr = np.r_[fpr[inside], limit]
    clippedPro = np.r_[pro[inside], np.interp(limit, fpr, pro)]
    return float(auc(clippedFpr, clippedPro) / limit), 'available'

def loadMasks(evaluations, category):
    masks = np.zeros((len(evaluations), imageSize, imageSize), dtype=np.uint8)
    reasons = []
    logStage('START', 'public mask alignment', category, 'all', 0, len(evaluations))
    for processed, (group, imagePath) in enumerate(evaluations, 1):
        if group == 'bad':
            maskPath = imagePath.parent.parent / 'ground_truth' / 'bad' / (imagePath.stem + '_mask.png')
            try:
                with Image.open(imagePath) as image:
                    originalSize = image.size
                with Image.open(maskPath) as mask:
                    mask.load()
                    nativeMask = np.asarray(mask)
                    if mask.size != originalSize or nativeMask.ndim != 2 or not np.isin(nativeMask, [0, 255]).all() or not nativeMask.any():
                        raise ValueError('mask is empty, nonbinary, or inconsistent with the original image')
                    resized = np.asarray(mask.resize((imageSize, imageSize), Image.Resampling.NEAREST)) > 0
                    if not resized.any():
                        # Diagnose loss; do not change labels, masks, selected identities, or denominators.
                        diagnostics = dict(category=category, relativeImagePath=imagePath.relative_to(datasetRoot).as_posix(),
                                           relativeMaskPath=maskPath.relative_to(datasetRoot).as_posix(),
                                           imageSizeWH=list(originalSize), nativeMaskSizeWH=list(mask.size),
                                           evaluationSizeWH=[imageSize, imageSize], resampling='Pillow NEAREST')
                        for name, array in [('native', nativeMask), ('resized', resized)]:
                            ys, xs = np.nonzero(array)
                            diagnostics[name] = dict(values=np.unique(array).tolist(), nonzeroArea=int(len(xs)),
                                                     boundingBoxXYXY=[int(xs.min()), int(ys.min()), int(xs.max()+1), int(ys.max()+1)] if len(xs) else None)
                        summary['maskDiagnostics'].append(diagnostics)
                        raise ValueError(f'anomaly disappears under NEAREST at {imageSize}x{imageSize}: native area={np.count_nonzero(nativeMask)}, resized area=0; maskDiagnostics records alignment details')
                    masks[processed-1] = resized
            except Exception as error:
                reasons.append(f'{imagePath.name}: {type(error).__name__}: {error}')
        if processed % 5 == 0:
            logStage('PROGRESS', 'public mask alignment', category, 'all', processed, len(evaluations))
    logStage('COMPLETE', 'public mask alignment', category, 'all', len(evaluations), len(evaluations))
    return masks, reasons

experimentsFinished = False
logStage('START', 'experiments')
try:
    for category in categories:
        references = selectedData[category]['reference']
        evaluations = selectedData[category]['evaluation']
        labels = np.array([int(group == 'bad') for group, _ in evaluations])
        bothClasses = set(labels.tolist()) == {0, 1}
        masks, maskReasons = loadMasks(evaluations, category)
        summary['limitations'].extend(f'{category}: {reason}' for reason in maskReasons)
        for condition in conditions:
            conditionStarted = time.perf_counter()
            logStage('START', 'condition', category, condition)
            imageValue, proValue = None, None
            imageStatus = proStatus = 'unavailable: at least two normal references and one evaluation image required'
            if len(references) >= 2 and evaluations:
                mean, precision = fitPadim(references, category, condition)
                scoreMaps = np.empty((len(evaluations), imageSize, imageSize), dtype=np.float32)
                logStage('START', 'inference (enhancement/features/scoring)', category, condition, 0, len(evaluations))
                for start in range(0, len(evaluations), batchSize):
                    batch = evaluations[start:start+batchSize]
                    difference = extractFeatures([p for _, p in batch], condition) - mean
                    distances = np.sqrt(np.maximum(0, np.einsum('npd,pde,npe->np', difference, precision, difference, optimize=True)))
                    maps = F.interpolate(torch.from_numpy(distances.reshape(-1, 1, featureGrid, featureGrid)),
                                         size=(imageSize, imageSize), mode='bilinear', align_corners=False).numpy()[:, 0]
                    for offset, scoreMap in enumerate(maps):
                        scoreMaps[start+offset] = gaussian_filter(scoreMap, sigma=4.0)
                    processed = start + len(batch)
                    if processed % 5 == 0:
                        logStage('PROGRESS', 'inference (enhancement/features/scoring)', category, condition, processed, len(evaluations))
                logStage('COMPLETE', 'inference (enhancement/features/scoring)', category, condition, len(evaluations), len(evaluations))
                logStage('START', 'metric evaluation', category, condition)
                if not np.isfinite(scoreMaps).all():
                    raise ValueError(f'{category}/{condition}: nonfinite anomaly scores')
                imageStatus = proStatus = 'unavailable: evaluation requires both normal and anomaly classes'
                if bothClasses:
                    imageValue = float(roc_auc_score(labels, scoreMaps.max(axis=(1, 2))))
                    imageStatus = 'available'
                    if maskReasons:
                        proStatus = 'unavailable: one or more selected public anomaly masks are missing or invalid; see summary limitations'
                    else:
                        proValue, proStatus = auPro005(masks, scoreMaps)
                del scoreMaps, mean, precision
                logStage('COMPLETE', 'metric evaluation', category, condition)
            for metricName, value, status in [('image_AUROC', imageValue, imageStatus), ('AU-PRO0.05', proValue, proStatus)]:
                metricRows.append(dict(dataset='MVTec AD 2', category=category, condition=condition, workloadMode=workloadSize,
                                       referenceCount=len(references), evaluationCount=len(evaluations),
                                       normalEvaluationCount=int((labels == 0).sum()), anomalyEvaluationCount=int((labels == 1).sum()),
                                       metricName=metricName, metricValue=value, metricStatus=status,
                                       runtimeSeconds=time.perf_counter()-conditionStarted))
                summary['metricAvailability'].append(dict(category=category, condition=condition, metric=metricName, status=status))
            logStage('COMPLETE', 'condition', category, condition)
            saveEvidence()  # Preserve observed timings and completed metrics; run remains incomplete.
        del masks
    experimentsFinished = True
except BaseException as error:
    summary['errors'].append(f'Experiments: {type(error).__name__}: {error}')
    saveEvidence()
    raise
logStage('COMPLETE', 'experiments')


# 6 — Results and Evidence
Only after all selected category/condition configurations (32 by default) have been visited is execution marked **complete**. Unsupported metrics stay blank with reasons; complete execution is not a claim that every metric was available or that enhancement helped. `metrics.csv`, `experiment_summary.json`, and `sample_manifest.csv` remain in the mode-specific UTC run folder. Conditions share identities, model, feature subset, and evaluation policy. A disconnect leaves the current run incomplete and preserves prior run folders; restart creates a new run rather than overwriting evidence. Model checkpoints/resume are outside Batch 1.
Carlos executes on the real datasets; Sam reviews the evidence; Carlos accepts. Synthetic implementation tests establish no real research result or statistical significance.
This all-eight small run is exploratory screening. Do not choose final categories or declare enhancement superior from this run alone; Team review determines any follow-up scope.


In [ ]:
# Finalize only if scoring/evaluation reached the explicit end-of-run gate.
logStage('START', 'results')
if not experimentsFinished or len(metricRows) != len(categories) * len(conditions) * 2:
    saveEvidence()
    raise RuntimeError('Incomplete experiments: refusing to mark the run complete.')

for row in metricRows:
    value = '' if row['metricValue'] is None else f"{row['metricValue']:.4f}"
    print(f"{row['category']}/{row['condition']} | {row['metricName']}={value} | {row['metricStatus']}", flush=True)
logStage('COMPLETE', 'results')
saveEvidence()  # All computation persisted; still incomplete until the final notebook gate.
logStage('COMPLETE', 'notebook')
summary['status'] = 'complete'
summary['totalRuntimeSeconds'] = time.perf_counter() - runStarted
temporaryPath = runRoot / 'experiment_summary.json.tmp'
temporaryPath.write_text(json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')
temporaryPath.replace(runRoot / 'experiment_summary.json')
print('Evidence:', runRoot, flush=True)
